# Multi-Horizon Forecasting for Limit Order Books: Novel Deep Learning Approaches and Hardware Acceleration using Intelligent Processing Units
### Authors: Zihao Zhang and Stefan Zohren
### Oxford-Man Institute of Quantitative Finance, Department of Engineering Science, University of Oxford

This jupyter notebook is used to demonstrate our recent paper [2] published in <...>. We use FI-2010 [1] dataset and present how model architecture is constructed here. The FI-2010 is publicly avilable and interested readers can check out their paper [1]. 

### Data:
The FI-2010 is publicly avilable and interested readers can check out their paper [1]. The dataset can be downloaded from: https://etsin.fairdata.fi/dataset/73eb48d7-4dbc-4a10-a52a-da745b47a649 

Otherwise, the notebook will download the data automatically or it can be obtained from: 

https://drive.google.com/drive/folders/1Xen3aRid9ZZhFqJRgEMyETNazk02cNmv?usp=sharing.

### References:

[1] Ntakaris A, Magris M, Kanniainen J, Gabbouj M, Iosifidis A. Benchmark dataset for mid‐price forecasting of limit order book data with machine learning methods. Journal of Forecasting. 2018 Dec;37(8):852-66. https://arxiv.org/abs/1705.03233

[2] Zhang Z, Zohren S. Multi-Horizon Forecasting for Limit Order Books: Novel Deep Learning Approaches and Hardware Acceleration using Intelligent Processing Units. 

#### This notebook demonstrates how to train DeepLOB-Seq2Seq by using tensorflow 2 on GPUs.

In [1]:
# obtain data
import os 
if not os.path.isfile('data.zip'):
    !wget https://raw.githubusercontent.com/zcakhaa/DeepLOB-Deep-Convolutional-Neural-Networks-for-Limit-Order-Books/master/data/data.zip
    !unzip -n data.zip
    print('data downloaded.')
else:
    print('data already existed.')

data already existed.


In [2]:
import torch
import os
import logging
import glob
import argparse
import sys
import time
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split

# Set random seeds
np.random.seed(1)
torch.manual_seed(2)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Optional: If you want to set memory growth for GPUs in PyTorch
if device.type == 'cuda':
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True
    # If you want to set memory growth, similar to TensorFlow:
    # torch.cuda.set_memory_growth(torch.cuda.current_device(), True)

# Optionally print information about available GPUs
if device.type == 'cuda':
    print(torch.cuda.device_count(), "CUDA device(s) available.")
    print("Current CUDA device:", torch.cuda.current_device())

# Example: How to print total and available GPU memory
# if torch.cuda.is_available():
#     print(torch.cuda.get_device_name(0))
#     print('Memory Usage:')
#     print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
#     print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

# Rest of your code follows here...


Pre process functions

In [3]:
import SL_preprocess as pp

Model

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [5]:
class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Encoder, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=input_dim[2], out_channels=32, kernel_size=(1, 2), stride=(1, 2))
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4, 1), padding=(2, 0)) # Adjusted padding
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4, 1), padding=(2, 0))

        self.conv4 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(1, 2), stride=(1, 2))
        self.conv5 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4, 1), padding=(2, 0))
        self.conv6 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4, 1), padding=(2, 0))

        self.conv7 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(1, 10))
        self.conv8 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4, 1), padding=(2, 0))
        self.conv9 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4, 1), padding=(2, 0))

        self.conv10 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(1, 1), padding=(0, 0))
        self.conv11 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 1), padding=(1, 0))

        self.conv12 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(1, 1), padding=(0, 0))
        self.conv13 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(5, 1), padding=(2, 0))

        self.pool = nn.MaxPool2d(kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
        self.conv14 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(1, 1), padding=(0, 0))

        self.lstm = nn.LSTM(input_size=192, hidden_size=latent_dim, batch_first=True) # Adjusted LSTM input size

    def forward(self, x):
        x = F.leaky_relu(self.conv1(x), negative_slope=0.01)
        x = F.leaky_relu(self.conv2(x), negative_slope=0.01)
        x = F.leaky_relu(self.conv3(x), negative_slope=0.01)

        x = F.leaky_relu(self.conv4(x), negative_slope=0.01)
        x = F.leaky_relu(self.conv5(x), negative_slope=0.01)
        x = F.leaky_relu(self.conv6(x), negative_slope=0.01)

        x = F.leaky_relu(self.conv7(x), negative_slope=0.01)
        x = F.leaky_relu(self.conv8(x), negative_slope=0.01)
        x = F.leaky_relu(self.conv9(x), negative_slope=0.01)

        convsecond_1 = F.leaky_relu(self.conv10(x), negative_slope=0.01)
        convsecond_1 = F.leaky_relu(self.conv11(convsecond_1), negative_slope=0.01)

        convsecond_2 = F.leaky_relu(self.conv12(x), negative_slope=0.01)
        convsecond_2 = F.leaky_relu(self.conv13(convsecond_2), negative_slope=0.01)

        convsecond_3 = self.pool(x)
        convsecond_3 = F.leaky_relu(self.conv14(convsecond_3), negative_slope=0.01)

        x = torch.cat([convsecond_1, convsecond_2, convsecond_3], dim=1) # This line was added0
        x = x.view(x.size(0), x.size(2), x.size(1))  # Reshape for LSTM

        output, (state_h, state_c) = self.lstm(x)
        return output, state_h, state_c
    

class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super(Decoder, self).__init__()
        self.lstm = nn.LSTM(input_size=latent_dim + 3, hidden_size=latent_dim, batch_first=True)
        self.fc = nn.Linear(latent_dim, 3)

    def forward(self, x, hidden):
        lstm_out, hidden = self.lstm(x, hidden)
        output = self.fc(lstm_out)
        return output, hidden



class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, input_tensor, target_length):
        enc_output, state_h, state_c = self.encoder(input_tensor)
        hidden = (state_h, state_c)
        
        # Initialize decoder's input with a tensor of zeros
        decoder_input = torch.zeros(input_tensor.size(0), 1, 3, device=input_tensor.device)
        decoder_input = torch.cat([decoder_input, enc_output[:, -1:, :]], dim=2)
        
        all_outputs = []
        
        for _ in range(target_length):
            output, hidden = self.decoder(decoder_input, hidden)
            all_outputs.append(output)
            decoder_input = torch.cat([output, enc_output[:, -1:, :]], dim=2)
        
        decoder_outputs = torch.cat(all_outputs, dim=1)
        return decoder_outputs



Just for testing

In [6]:
## Example usage of the Seq2Seq model
#if __name__ == "__main__":
#    # Define input dimensions and latent dimensions
#    input_dim = (50, 40, 1)  # Example: (height, width, channels)
#    latent_dim = 128
#
#    # Instantiate the encoder and decoder
#    encoder = Encoder(input_dim=input_dim, latent_dim=latent_dim)
#    decoder = Decoder(latent_dim=latent_dim)
#
#    # Instantiate the Seq2Seq model
#    seq2seq = Seq2Seq(encoder, decoder)
#    
#    # Create a random input tensor with shape (batch_size, channels, height, width)
#    batch_size = 2
#    input_tensor = torch.randn(batch_size, input_dim[2], input_dim[0], input_dim[1])
#    
#    # Define the target length for the output sequence
#    target_length = 5
#    
#    # Forward pass through the Seq2Seq model
#    output = seq2seq(input_tensor, target_length)
#    
#    # Print the shape of the final output
#    print("Seq2Seq model output shape:", output.shape)

Run

In [7]:
# Define constants and paths
data_path = './data'  # Replace with your local path
T = 5 # i herits from the test set  
batch_size = 2056 # try 32
n_hidden = 64
checkpoint_filepath = './model_deeplob_seq/weights'

# Load data from files
dec_train = np.loadtxt(os.path.join('Train_Dst_NoAuction_DecPre_CF_7.txt'))
dec_test1 = np.loadtxt(os.path.join('Test_Dst_NoAuction_DecPre_CF_7.txt'))
dec_test2 = np.loadtxt(os.path.join('Test_Dst_NoAuction_DecPre_CF_8.txt'))
dec_test3 = np.loadtxt(os.path.join('Test_Dst_NoAuction_DecPre_CF_9.txt'))
dec_test = np.hstack((dec_test1, dec_test2, dec_test3))

# Preprocess data
train_lob = pp.prepare_x(dec_train)
test_lob = pp.prepare_x(dec_test)

train_label = pp.get_label(dec_train)
test_label = pp.get_label(dec_test)




In [8]:
train_encoder_input, train_decoder_target = pp.data_classification(train_lob, train_label, T)


In [9]:
train_decoder_input = pp.prepare_decoder_input(train_encoder_input, teacher_forcing=False)



In [10]:
test_encoder_input, test_decoder_target = pp.data_classification(test_lob, test_label, T)


In [11]:
test_decoder_input = pp.prepare_decoder_input(test_encoder_input, teacher_forcing=False)


In [12]:
print(f'train_encoder_input.shape = {train_encoder_input.shape},\n'
      f'train_decoder_target.shape = {train_decoder_target.shape}')
print(f'test_encoder_input.shape = {test_encoder_input.shape},\n'
      f'test_decoder_target.shape = {test_decoder_target.shape}')

train_encoder_input.shape = (254746, 5, 40, 1),
train_decoder_target.shape = (254746, 5, 3)
test_encoder_input.shape = (139583, 5, 40, 1),
test_decoder_target.shape = (139583, 5, 3)


In [13]:
# Convert numpy arrays to PyTorch tensors
train_encoder_input_tensor = torch.tensor(train_encoder_input, dtype=torch.float32)
train_decoder_input_tensor = torch.tensor(train_decoder_input, dtype=torch.float32)
train_decoder_target_tensor = torch.tensor(train_decoder_target, dtype=torch.float32)

test_encoder_input_tensor = torch.tensor(test_encoder_input, dtype=torch.float32)
test_decoder_input_tensor = torch.tensor(test_decoder_input, dtype=torch.float32)
test_decoder_target_tensor = torch.tensor(test_decoder_target, dtype=torch.float32)

# Permute in orde to have the correct shape
train_encoder_input_tensor = train_encoder_input_tensor.permute(0,3,1,2)
test_encoder_input_tensor = test_encoder_input_tensor.permute(0,3,1,2)





# Create DataLoader for training and testing datasets
train_dataset = TensorDataset(train_encoder_input_tensor, train_decoder_input_tensor, train_decoder_target_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(test_encoder_input_tensor, test_decoder_input_tensor, test_decoder_target_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f'train_encoder_input_tensor.shape = {train_encoder_input_tensor.shape}, \n'
      f'train_decoder_target_tensor.shape = {train_decoder_target_tensor.shape}')
print(f'test_encoder_input_tensor.shape = {test_encoder_input_tensor.shape}, \n'
      f'test_decoder_target_tensor.shape = {test_decoder_target_tensor.shape}')




train_encoder_input_tensor.shape = torch.Size([254746, 1, 5, 40]), 
train_decoder_target_tensor.shape = torch.Size([254746, 5, 3])
test_encoder_input_tensor.shape = torch.Size([139583, 1, 5, 40]), 
test_decoder_target_tensor.shape = torch.Size([139583, 5, 3])


In [14]:
# Instantiate your Seq2Seq model
input_dim = (train_encoder_input_tensor.shape[2], train_encoder_input_tensor.shape[3], train_encoder_input_tensor.shape[1])
print(input_dim)
print(input_dim[2])
print(n_hidden) # latent_dim

encoder = Encoder(input_dim=input_dim, latent_dim=n_hidden)
decoder = Decoder(latent_dim=n_hidden)
model = Seq2Seq(encoder=encoder, decoder=decoder)

# Define loss function, optimizer, and metrics
criterion = nn.CrossEntropyLoss() # CrossEntropyLoss expects target labels, not one-hot encoded targets
optimizer = optim.Adam(model.parameters())

# Optionally, define metrics (accuracy in this case)
def accuracy(outputs, targets):
    # Convert outputs to predicted class labels
    _, predicted = torch.max(outputs, 2)  # Shape: [batch_size, seq_len]
    
    # Convert targets to true class labels
    _, true = torch.max(targets, 2)  # Shape: [batch_size, seq_len]
    
    # Calculate the number of correct predictions
    correct = (predicted == true).sum().item()
    
    # Calculate the total number of predictions
    total = true.numel()
    
    # Calculate accuracy
    acc = correct / total
    return acc


def custom_cross_entropy_loss(predictions, targets):
    # Flatten predictions and targets to shape (batch_size * seq_len, num_classes)
    predictions = predictions.view(-1, predictions.size(-1))
    targets = targets.view(-1, targets.size(-1))
    
    # Apply log_softmax to predictions and sum up over the classes
    log_preds = F.log_softmax(predictions, dim=-1)
    loss = -torch.sum(targets * log_preds, dim=-1)
    
    # Return the mean loss over the batch
    return loss.mean()

# Example usage:
# loss = custom_cross_entropy_loss(outputs, decoder_targets)


# Split the dataset into training and validation sets
split_train_val = int(np.floor(len(train_encoder_input) * 0.8))

# Create DataLoader for validation set
val_dataset = TensorDataset(train_encoder_input_tensor[split_train_val:], train_decoder_input_tensor[split_train_val:], train_decoder_target_tensor[split_train_val:])
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

(5, 40, 1)
1
64


In [15]:
# Define training parameters
epochs = 5
best_val_loss = float('inf')
best_model_path = checkpoint_filepath + '_best.pt'

# Move model to device (GPU if available)
model = model.to(device)


    
# Iterate over training batches
for batch_idx, (encoder_inputs, decoder_inputs, decoder_targets) in enumerate(train_loader):
    encoder_inputs, decoder_inputs, decoder_targets = encoder_inputs.to(device), decoder_inputs.to(device), decoder_targets.to(device)
    optimizer.zero_grad()
    
    # Forward pass
    print(f"  Batch [{batch_idx+1}/{len(train_loader)}]")
    #print(f"    Encoder inputs shape: {encoder_inputs.shape}")
    #print(f"    Target length: {T}")
    outputs = model(encoder_inputs, target_length=T)
    
    print(outputs.shape)   # torch.Size([32, 5, 3])
    print(decoder_targets.shape)   # torch.Size([32, 5, 3])

    # Calculate training accuracy
    acc = accuracy(outputs, decoder_targets)
    print(acc)
        
    break
    
   

  Batch [1/124]
torch.Size([2056, 5, 3])
torch.Size([2056, 5, 3])
0.3147859922178988


In [16]:
# Define training parameters
epochs = 5
best_val_loss = float('inf')
best_model_path = checkpoint_filepath + '_best.pt'

# Move model to device (GPU if available)
model = model.to(device)

# Training loop
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_acc = 0.0
    
    print(f"Epoch [{epoch+1}/{epochs}]")
    
    # Iterate over training batches
    for batch_idx, (encoder_inputs, decoder_inputs, decoder_targets) in enumerate(train_loader):
        encoder_inputs, decoder_inputs, decoder_targets = encoder_inputs.to(device), decoder_inputs.to(device), decoder_targets.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        print(f"  Batch [{batch_idx+1}/{len(train_loader)}]")
        #print(f"    Encoder inputs shape: {encoder_inputs.shape}")
        #print(f"    Target length: {T}")
        outputs = model(encoder_inputs, target_length=T)

        #print(outputs.shape)
        #print(decoder_targets.shape)
        
        # Calculate loss
        loss = custom_cross_entropy_loss(outputs, decoder_targets)
        train_loss += loss.item()
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Calculate training accuracy
        acc = accuracy(outputs, decoder_targets)
        train_acc += acc
        
        print(f"    Loss: {loss.item():.4f}, Acc: {acc:.4f}")
    
    # Average loss and accuracy over batches
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    
    # Validation loop
    model.eval()
    val_loss = 0.0
    val_acc = 0.0
    
    with torch.no_grad():
        for encoder_inputs, decoder_inputs, decoder_targets in val_loader:
            encoder_inputs, decoder_inputs, decoder_targets = encoder_inputs.to(device), decoder_inputs.to(device), decoder_targets.to(device)
            
            # Forward pass
            outputs = model(encoder_inputs, target_length=T)
            
            # Calculate loss
            loss = custom_cross_entropy_loss(outputs, decoder_targets)
            val_loss += loss.item()
            
            # Calculate validation accuracy
            acc = accuracy(outputs, decoder_targets)
            val_acc += acc
    
    # Average loss and accuracy over validation batches
    val_loss /= len(val_loader)
    val_acc /= len(val_loader)
    
    # Print epoch results
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
    
    # Save the best model based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        #torch.save(model.state_dict(), best_model_path)
        print(f'  Saving the best model with validation loss: {best_val_loss:.4f}')

    print()  # Empty line for better readability between epochs


Epoch [1/5]
  Batch [1/124]
    Loss: 1.1040, Acc: 0.3063
  Batch [2/124]
    Loss: 1.1014, Acc: 0.2992
  Batch [3/124]
    Loss: 1.0973, Acc: 0.2953
  Batch [4/124]
    Loss: 1.0950, Acc: 0.3795
  Batch [5/124]
    Loss: 1.0899, Acc: 0.4053
  Batch [6/124]
    Loss: 1.0877, Acc: 0.4011
  Batch [7/124]
    Loss: 1.0841, Acc: 0.4050
  Batch [8/124]
    Loss: 1.0829, Acc: 0.3961
  Batch [9/124]
    Loss: 1.0731, Acc: 0.4166
  Batch [10/124]
    Loss: 1.0743, Acc: 0.4038
  Batch [11/124]
    Loss: 1.0702, Acc: 0.4106
  Batch [12/124]
    Loss: 1.0700, Acc: 0.4015
  Batch [13/124]
    Loss: 1.0670, Acc: 0.4058
  Batch [14/124]
    Loss: 1.0641, Acc: 0.4009
  Batch [15/124]
    Loss: 1.0555, Acc: 0.4121
  Batch [16/124]
    Loss: 1.0568, Acc: 0.4452
  Batch [17/124]
    Loss: 1.0505, Acc: 0.4545
  Batch [18/124]
    Loss: 1.0502, Acc: 0.4441
  Batch [19/124]
    Loss: 1.0532, Acc: 0.4509
  Batch [20/124]
    Loss: 1.0502, Acc: 0.4490
  Batch [21/124]
    Loss: 1.0488, Acc: 0.4270
  Batch [2

In [ ]:
split_train_val = int(np.floor(len(train_encoder_input) * 0.8))

model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    mode='auto',
    save_best_only=True)

model.fit([train_encoder_input[:split_train_val], train_decoder_input[:split_train_val]], 
          train_decoder_target[:split_train_val],
          validation_data=([train_encoder_input[split_train_val:], train_decoder_input[split_train_val:]], 
          train_decoder_target[split_train_val:]),
          epochs=epochs, batch_size=batch_size, verbose=2, callbacks=[model_checkpoint_callback])

Epoch 1/50
6368/6368 - 181s - loss: 0.9576 - accuracy: 0.5171 - val_loss: 0.8496 - val_accuracy: 0.6138
Epoch 2/50
6368/6368 - 179s - loss: 0.6975 - accuracy: 0.7035 - val_loss: 0.7852 - val_accuracy: 0.6524
Epoch 3/50
6368/6368 - 180s - loss: 0.6446 - accuracy: 0.7311 - val_loss: 0.7602 - val_accuracy: 0.6728
Epoch 4/50
6368/6368 - 179s - loss: 0.6162 - accuracy: 0.7455 - val_loss: 0.7625 - val_accuracy: 0.6723
Epoch 5/50
6368/6368 - 180s - loss: 0.5984 - accuracy: 0.7540 - val_loss: 0.7443 - val_accuracy: 0.6825
Epoch 6/50
6368/6368 - 181s - loss: 0.5848 - accuracy: 0.7603 - val_loss: 0.7305 - val_accuracy: 0.6892
Epoch 7/50
6368/6368 - 180s - loss: 0.5740 - accuracy: 0.7649 - val_loss: 0.7427 - val_accuracy: 0.6822
Epoch 8/50
6368/6368 - 180s - loss: 0.5650 - accuracy: 0.7689 - val_loss: 0.7615 - val_accuracy: 0.6819
Epoch 9/50
6368/6368 - 179s - loss: 0.5576 - accuracy: 0.7719 - val_loss: 0.7527 - val_accuracy: 0.6846
Epoch 10/50
6368/6368 - 179s - loss: 0.5518 - accuracy: 0.7745 -

In [ ]:
model.load_weights(checkpoint_filepath)
pred = model.predict([test_encoder_input, test_decoder_input])

In [ ]:
evaluation_metrics(test_decoder_target, pred)


Prediction horizon = 0
accuracy_score = 0.8222921354756411
classification_report =               precision    recall  f1-score   support

           0     0.6910    0.5771    0.6289     21147
           1     0.8593    0.9309    0.8937     98624
           2     0.7136    0.5427    0.6165     19767

    accuracy                         0.8223    139538
   macro avg     0.7546    0.6835    0.7130    139538
weighted avg     0.8131    0.8223    0.8143    139538

-------------------------------
Prediction horizon = 1
accuracy_score = 0.7423497541888231
classification_report =               precision    recall  f1-score   support

           0     0.6292    0.5130    0.5652     27448
           1     0.7868    0.8924    0.8363     86605
           2     0.6454    0.4793    0.5501     25485

    accuracy                         0.7423    139538
   macro avg     0.6871    0.6283    0.6505    139538
weighted avg     0.7300    0.7423    0.7307    139538

-------------------------------
Predicti